In [1]:
import pandas as pd
import numpy as np


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [4]:
np.random.seed(42)
n_samples = 1000

In [5]:
data = pd.DataFrame({
    "fill_level": np.random.randint(10, 101, n_samples),       # Percentage
    "weight_kg": np.random.uniform(2, 50, n_samples),          # Waste weight
    "temperature": np.random.uniform(20, 45, n_samples),       # Temperature
    "days_since_collection": np.random.randint(0, 11, n_samples),
    "odor_level": np.random.randint(1, 11, n_samples),         # 1-10
    "population_nearby": np.random.randint(10, 500, n_samples)
})

In [6]:
data["collection_required"] = (
    (data["fill_level"] >= 75) |
    (data["weight_kg"] >= 35) |
    (data["days_since_collection"] >= 7) |
    (data["odor_level"] >= 8)
).astype(int)

print("First 5 records:")
print(data.head())

First 5 records:
   fill_level  weight_kg  temperature  days_since_collection  odor_level  \
0          61  36.636634    24.444160                      5           4   
1          24   4.308543    37.290451                     10           6   
2          81  39.512695    35.451292                      3           9   
3          70  41.741163    35.829741                      7           2   
4          30  38.024092    25.722365                      2           7   

   population_nearby  collection_required  
0                484                    1  
1                317                    1  
2                170                    1  
3                 10                    1  
4                245                    1  


In [7]:
X = data.drop("collection_required", axis=1)
y = data["collection_required"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [9]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [10]:
y_pred = model.predict(X_test)

In [11]:
accuracy = accuracy_score(y_test, y_pred)

In [12]:
print("\nModel Accuracy:", round(accuracy * 100, 2), "%")


Model Accuracy: 100.0 %


In [13]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        46
           1       1.00      1.00      1.00       154

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



In [14]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Confusion Matrix:
[[ 46   0]
 [  0 154]]


In [15]:
new_bin = pd.DataFrame({
    "fill_level": [85],
    "weight_kg": [38],
    "temperature": [32],
    "days_since_collection": [6],
    "odor_level": [7],
    "population_nearby": [250]
})

In [16]:
prediction = model.predict(new_bin)

In [17]:
if prediction[0] == 1:
    print("\nPrediction: COLLECTION REQUIRED")
else:
    print("\nPrediction: COLLECTION NOT REQUIRED")


Prediction: COLLECTION REQUIRED


In [18]:
probability = model.predict_proba(new_bin)[0][1]

In [19]:
print(
    "Probability that collection is required:",
    round(probability * 100, 2),
    "%"
)

Probability that collection is required: 100.0 %


In [20]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

In [21]:
print("\nFeature Importance:")
print(feature_importance)


Feature Importance:
                 Feature  Importance
3  days_since_collection    0.287595
1              weight_kg    0.244014
4             odor_level    0.229406
0             fill_level    0.187048
5      population_nearby    0.027689
2            temperature    0.024247
